In [ ]:
    from pathlib import Path
    import pandas as pd


    SCHEMA_DIR = Path("../../data/processed/schema_csvs")


    directorship = pd.read_csv(SCHEMA_DIR / "Directorship.csv")
    company = pd.read_csv(SCHEMA_DIR / "Company.csv")
    ownership = pd.read_csv(SCHEMA_DIR / "Ownership.csv")

In [ ]:
company_ml = company.copy()

def extract_id(x):
    if pd.isna(x):
        return None
    return str(x).strip("[]'\"")

ownership["owner_clean"] = ownership["properties.owner"].apply(extract_id)
ownership["asset_clean"] = ownership["properties.asset"].apply(extract_id)

In [ ]:
ownership_features = (
    ownership.groupby("asset_clean")
    .agg(
        num_owners=("owner_clean", "nunique"),
        ownership_links=("owner_clean", "count")    
    )
    .reset_index()
    .rename(columns={"asset_clean": "id"})
)

In [ ]:
directorship["director_clean"] = directorship["properties.director"].apply(extract_id)
directorship["company_clean"] = directorship["properties.organization"].apply(extract_id)

In [ ]:
directorship_features = (
    directorship.groupby("company_clean")
    .agg(
        num_directors=("director_clean", "nunique"),
        director_links=("director_clean", "count")
    )
    .reset_index()
    .rename(columns={"company_clean": "id"})
)

In [ ]:
company_ml = company_ml.merge(ownership_features, on="id", how="left")
company_ml = company_ml.merge(directorship_features, on="id", how="left")

# Fill missing values
for col in ["num_owners", "ownership_links", "num_directors", "director_links"]:
    if col in company_ml.columns:
        company_ml[col] = company_ml[col].fillna(0)

In [ ]:
cols = [
    "id",
    "target",
    "properties.country",
    "properties.jurisdiction",
    "properties.sector",
    "properties.topics",
    "properties.status",
    "properties.legalForm",
    "num_owners",
    "ownership_links",
    "num_directors",
    "director_links"
]

company_ml = company_ml[[c for c in cols if c in company_ml.columns]]

In [ ]:
def clean_list_column(x):
    if pd.isna(x):
        return None
    x = str(x).strip("[]")
    x = x.replace('"', '').replace("'", "")
    return x.split(";")[0].strip()

for col in ["properties.country", "properties.jurisdiction", "properties.sector""properties.topics","properties.status","properties.legalForm"]:
    if col in company_ml.columns:
        company_ml[col] = company_ml[col].apply(clean_list_column)

In [ ]:
company_ml["properties.country"] = company_ml["properties.country"].fillna("Unknown")
company_ml["properties.jurisdiction"] = company_ml["properties.jurisdiction"].fillna("Unknown")
company_ml["properties.sector"] = company_ml["properties.sector"].fillna("Unknown")

company_ml["target"] = company_ml["target"].astype(int)

In [ ]:
print(company_ml.head())
print(company_ml.shape)

In [ ]:
print(company_ml.isna().sum())
print(company_ml["target"].value_counts())
print(company_ml.describe(include="all"))

In [ ]:

print("\nTopics distribution:")
print(company_ml["properties.topics"].value_counts().head(20))

print("\nStatus distribution:")
print(company_ml["properties.status"].value_counts().head(20))

print("\nLegal form distribution:")
print(company_ml["properties.legalForm"].value_counts().head(20))

In [ ]:
sanction = pd.read_csv(SCHEMA_DIR / "Sanction.csv")




In [ ]:
print(sanction.shape)
print(sanction.columns.tolist())
sanction.head(20).T

In [ ]:
"""
CSV Explorer — OpenSanctions Schema Files
==========================================
Run this script and paste the output back.
It will show us:
  - Exact column names
  - Sample values (so we can see the format of IDs, topics, target etc.)
  - Non-null counts (so we know what's actually populated)
  - Unique value counts for categorical columns
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
SCHEMA_DIR = Path("../../data/processed/schema_csvs")  # adjust if needed

FILES_TO_EXPLORE = [
    "Company.csv",
    "Directorship.csv",
    "Ownership.csv",
    "Membership.csv",
    "Representation.csv",
    "Sanction.csv",
    "Address.csv",
]
# ─────────────────────────────────────────────────────────────────────────────


def explore_csv(filename):
    path = SCHEMA_DIR / filename
    print(f"\n{'█'*65}")
    print(f"  {filename}")
    print(f"{'█'*65}")

    try:
        df = pd.read_csv(path, low_memory=False)
    except FileNotFoundError:
        print(f"  !! File not found at: {path}")
        return

    # ── Basic shape ──
    print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    # ── Column summary ──
    print(f"\n  {'COLUMN':<45} {'NON-NULL':>10} {'DTYPE':<12} {'UNIQUE':>8}")
    print(f"  {'-'*45} {'-'*10} {'-'*12} {'-'*8}")
    for col in df.columns:
        non_null  = df[col].notna().sum()
        pct       = non_null / len(df) * 100
        dtype     = str(df[col].dtype)
        n_unique  = df[col].nunique()
        print(f"  {col:<45} {non_null:>6,} ({pct:4.0f}%) {dtype:<12} {n_unique:>8,}")

    # ── Sample values for every column ──
    print(f"\n  SAMPLE VALUES (first 3 non-null per column):")
    print(f"  {'-'*65}")
    for col in df.columns:
        samples = df[col].dropna().head(3).tolist()
        print(f"  {col:<45} {samples}")

    # ── Special: show unique values if small cardinality ──
    print(f"\n  LOW-CARDINALITY COLUMNS (≤ 20 unique values):")
    print(f"  {'-'*65}")
    found_any = False
    for col in df.columns:
        n_unique = df[col].nunique()
        if 0 < n_unique <= 20:
            vals = df[col].value_counts().head(20).to_dict()
            print(f"  {col:<45} {vals}")
            found_any = True
    if not found_any:
        print("  (none)")

    # ── Special: always show 'target' column if it exists ──
    if "target" in df.columns:
        print(f"\n  TARGET COLUMN BREAKDOWN:")
        print(df["target"].value_counts(dropna=False).to_string())

    # ── Special: show any column with 'topic' in name ──
    topic_cols = [c for c in df.columns if "topic" in c.lower()]
    if topic_cols:
        for tc in topic_cols:
            print(f"\n  TOPICS COLUMN — '{tc}' value counts (top 20):")
            print(df[tc].value_counts(dropna=False).head(20).to_string())

    print()


# ── RUN ───────────────────────────────────────────────────────────────────────
for f in FILES_TO_EXPLORE:
    explore_csv(f)

print("\n" + "="*65)
print("  Done. Paste this entire output back to continue.")
print("="*65)

In [ ]:
print(company[company['target'] == True]['properties.topics'].value_counts().head(30))

In [2]:
"""
OpenSanctions — Clean Company Risk Dataset Builder
===================================================
Output: one CSV, one row per company, ready for any downstream use.

Columns produced:
  LABEL
    risk_tier                 : 0 = no flag, 1 = medium risk, 2 = high/direct sanction

  A. GEOGRAPHY
    country                   : best available country code (jurisdiction > mainCountry > country)
    is_high_risk_country      : 1 if country is in high-risk set
    country_missing           : 1 if no country found at all
    jurisdiction_country_mismatch : 1 if incorporation country ≠ operating country

  B. COMPANY TYPE
    legal_form_bucket         : llc_private / joint_stock_public / bank_financial /
                                state_enterprise / foundation_trust / partnership /
                                nonprofit_ngo / other / unknown
    has_sector                : 1 if sector field is populated
    is_holding                : 1 if company name contains holding/group/invest/capital etc.
    is_trading                : 1 if company name contains trading/import/export/commerce

  C. IDENTIFIER RICHNESS
    has_lei / has_duns / has_bic / has_vat / has_tax / has_reg / has_inn / has_ogrn
    identifier_count          : total number of identifiers present (0 = red flag)

  D. ALIAS COMPLEXITY
    alias_count               : number of aliases
    alias_script_diversity    : number of distinct scripts used across aliases
    has_weak_alias            : 1 if weakAlias field is populated
    has_previous_name         : 1 if previousName field is populated

  E. TEMPORAL
    incorporation_year        : year of incorporation
    company_age_years         : CURRENT_YEAR - incorporation_year
    incorporation_missing     : 1 if no incorporation date
    has_dissolution_date      : 1 if dissolution date is present

  F. DIRECTORSHIP (joined from Directorship.csv)
    director_count            : number of unique directors
    active_director_count     : directors with status 'Appointed' / 'ACTIVE'
    resigned_director_count   : directors with status 'Resigned'
    director_role_diversity   : number of unique roles across all directorships
    no_directors_found        : 1 if company has zero directorship records

  G. OWNERSHIP (joined from Ownership.csv)
    owner_count               : number of unique owners
    avg_ownership_pct         : average ownership percentage (where available)
    max_ownership_pct         : maximum ownership percentage (where available)
    has_majority_owner        : 1 if any owner holds > 50%
    ownership_pct_missing     : 1 if no ownership percentages available
    no_owners_found           : 1 if company has zero ownership records
"""

import ast
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
SCHEMA_DIR   = Path("../../data/processed/schema_csvs")
OUTPUT_PATH  = Path("../../data/processed/company_risk_dataset.csv")
CURRENT_YEAR = 2026

HIGH_RISK_COUNTRIES = {
    "ru", "ir", "kp", "sy", "by", "cu", "ve", "mm", "so", "sd",
    "ly", "ye", "zw", "cf", "cd", "iq", "lb", "ni", "af", "ml",
}

# Authoritative risk topics from FollowTheMoney source (topic.py RISKS set)
# https://followthemoney.tech/explorer/types/topic/#data-reference
# These are topics that "imply a counterparty risk in business dealings"
# Note: fin.bank, corp.offshore, corp.shell are deliberately excluded —
# they are descriptive taxonomy only, NOT risk indicators per FTM definition.
FTM_RISK_TOPICS = {
    "corp.disqual",
    "crime", "crime.boss", "crime.fin", "crime.fraud", "crime.terror",
    "crime.theft", "crime.traffick", "crime.war",
    "debarment",
    "export.control", "export.control.linked", "export.risk",
    "invest.risk",
    "mare.detained", "mare.shadow",
    "poi",
    "reg.action", "reg.warn",
    "role.oligarch", "role.pep", "role.rca",
    "sanction", "sanction.counter", "sanction.linked",
    "wanted",
}

LEGAL_FORM_BUCKETS = {
    "llc_private": [
        "llc", "ооо", "sarl", "gmbh", "srl", "lda", "bv", "kft", "sprl",
        "limited liability", "общество с ограниченной", "besloten",
        "société à responsabilité", "private limited", "pvt", "pte",
    ],
    "joint_stock_public": [
        "jsc", "ojsc", "pjsc", "ао ", "пао", " ao ", "акционерное общество",
        "joint stock", "open joint", " sa ", " ag ", "plc", " nv ", "nyrt",
        "public limited", "sociedade anónima",
    ],
    "bank_financial": [
        "bank", "банк", "credit", "кредит", "финансов", "insurance",
        "страхов", "investment fund", "инвестиционный фонд",
    ],
    "state_enterprise": [
        "state", "federal", "unitary", "фгуп", "гуп", "казенное",
        "государственн", "municipal", "муниципальн",
    ],
    "foundation_trust": [
        "foundation", "trust", "stiftung", "fondation", "fundación",
        "charity", "endowment",
    ],
    "partnership": [
        "llp", " lp ", "partnership", "коммандитн", "полное товарищество",
    ],
    "nonprofit_ngo": [
        "ngo", "association", "некоммерческ", "non-profit", "nonprofit",
        "cooperative", "кооператив",
    ],
}

HOLDING_KEYWORDS  = ["holding", "holdings", "group", "invest", "capital",
                      "international", "global", "ventures", "partners"]
TRADING_KEYWORDS  = ["trading", "trade", "import", "export", "commerce",
                      "commodities", "supply", "logistics"]
# ─────────────────────────────────────────────────────────────────────────────


# ═════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═════════════════════════════════════════════════════════════════════════════

def load(filename):
    path = SCHEMA_DIR / filename
    df = pd.read_csv(path, low_memory=False)
    print(f"  Loaded {filename}: {len(df):,} rows")
    return df


def parse_list_field(series):
    """'["a", "b"]' → actual Python list. Returns Series of lists."""
    def _parse(val):
        if pd.isna(val):
            return []
        try:
            result = ast.literal_eval(str(val))
            return result if isinstance(result, list) else [result]
        except Exception:
            return [str(val)]
    return series.apply(_parse)


def first_value(series):
    """First item from a stringified list column, lowercased. NaN if empty."""
    return parse_list_field(series).apply(
        lambda x: x[0].strip().lower() if x else np.nan
    )


def clean_id(series):
    """'["abc123"]' → 'abc123'"""
    return parse_list_field(series).apply(
        lambda x: x[0].strip() if x else np.nan
    )


def count_list_field(series):
    """Number of items in a stringified list column."""
    return parse_list_field(series).apply(len)


# ═════════════════════════════════════════════════════════════════════════════
# LOAD
# ═════════════════════════════════════════════════════════════════════════════
print("\n── Loading files ───────────────────────────────────────────")
company      = load("Company.csv")
directorship = load("Directorship.csv")
ownership    = load("Ownership.csv")


# ═════════════════════════════════════════════════════════════════════════════
# LABEL
# ═════════════════════════════════════════════════════════════════════════════
print("\n── Building label ──────────────────────────────────────────")

def assign_risk_tier(topics_val):
    if pd.isna(topics_val) or str(topics_val).strip() in ("", "[]", "nan"):
        return 0

    try:
        topics_list = ast.literal_eval(str(topics_val))
    except Exception:
        topics_list = [str(topics_val)]

    topics_set = {t.strip().lower() for t in topics_list}

    # Tier 2 — directly sanctioned:
    # 'sanction' present but NOT via 'sanction.linked' or 'sanction.counter'
    # (those are risk-adjacent, not direct listing)
    if "sanction" in topics_set and not {"sanction.linked", "sanction.counter"} & topics_set:
        return 2

    # Tier 1 — any other official FTM risk topic
    # Source: followthemoney/types/topic.py RISKS set
    if topics_set & FTM_RISK_TOPICS:
        return 1

    # Tier 0 — descriptive only (fin.bank, corp.offshore, corp.shell, gov.*, etc.)
    return 0

company["risk_tier"] = company["properties.topics"].apply(assign_risk_tier)
counts = company["risk_tier"].value_counts().sort_index()
print(f"  Tier 0 — no flag:     {counts.get(0, 0):,}")
print(f"  Tier 1 — medium risk: {counts.get(1, 0):,}")
print(f"  Tier 2 — high risk:   {counts.get(2, 0):,}")


# ═════════════════════════════════════════════════════════════════════════════
# INITIALISE FEATURE TABLE
# ═════════════════════════════════════════════════════════════════════════════
feat = pd.DataFrame({
    "id": company["id"],
    "target": company["target"],        # ← ADD THIS
    "risk_tier": company["risk_tier"]
})


# ═════════════════════════════════════════════════════════════════════════════
# A. GEOGRAPHY
# ═════════════════════════════════════════════════════════════════════════════
print("\n── A. Geography ────────────────────────────────────────────")

def get_best_country(row):
    for col in ["properties.jurisdiction", "properties.mainCountry", "properties.country"]:
        val = row.get(col)
        if pd.notna(val):
            parsed = parse_list_field(pd.Series([val]))[0]
            if parsed:
                return parsed[0].strip().lower()
    return np.nan

feat["country"]              = company.apply(get_best_country, axis=1)
feat["is_high_risk_country"] = feat["country"].isin(HIGH_RISK_COUNTRIES).astype(int)
feat["country_missing"]      = feat["country"].isna().astype(int)

jurisdiction = first_value(company["properties.jurisdiction"])
main_country = first_value(company["properties.mainCountry"])
feat["jurisdiction_country_mismatch"] = (
    jurisdiction.notna() & main_country.notna() & (jurisdiction != main_country)
).astype(int)


# ═════════════════════════════════════════════════════════════════════════════
# B. COMPANY TYPE
# ═════════════════════════════════════════════════════════════════════════════
print("\n── B. Company type ─────────────────────────────────────────")

# Legal form bucket
def bucket_legal_form(val):
    if pd.isna(val):
        return "unknown"
    text = str(val).lower()
    for bucket, keywords in LEGAL_FORM_BUCKETS.items():
        if any(kw in text for kw in keywords):
            return bucket
    return "other"

raw_legal_form        = first_value(company["properties.legalForm"])
feat["legal_form_bucket"] = raw_legal_form.apply(bucket_legal_form)
feat["has_sector"]        = company["properties.sector"].notna().astype(int)

# Name-based signals — use caption (always populated) as the name source
def name_contains(keywords):
    return (
        company["caption"]
        .fillna("")
        .str.lower()
        .apply(lambda n: int(any(kw in n for kw in keywords)))
    )

feat["is_holding"] = name_contains(HOLDING_KEYWORDS)
feat["is_trading"]  = name_contains(TRADING_KEYWORDS)


# ═════════════════════════════════════════════════════════════════════════════
# C. IDENTIFIER RICHNESS
# ═════════════════════════════════════════════════════════════════════════════
print("\n── C. Identifier richness ──────────────────────────────────")

ID_COLS = {
    "has_lei":  "properties.leiCode",
    "has_duns": "properties.dunsCode",
    "has_bic":  "properties.swiftBic",
    "has_vat":  "properties.vatCode",
    "has_tax":  "properties.taxNumber",
    "has_reg":  "properties.registrationNumber",
    "has_inn":  "properties.innCode",
    "has_ogrn": "properties.ogrnCode",
}

for feat_name, col in ID_COLS.items():
    feat[feat_name] = company[col].notna().astype(int) if col in company.columns else 0

feat["identifier_count"] = feat[list(ID_COLS.keys())].sum(axis=1)


# ═════════════════════════════════════════════════════════════════════════════
# D. ALIAS COMPLEXITY
# ═════════════════════════════════════════════════════════════════════════════
print("\n── D. Alias complexity ─────────────────────────────────────")

feat["alias_count"] = count_list_field(company["properties.alias"])

def count_scripts(val):
    if pd.isna(val):
        return 0
    text = str(val)
    scripts = [
        bool(re.search(r'[a-zA-Z]', text)),           # Latin
        bool(re.search(r'[\u0400-\u04FF]', text)),     # Cyrillic
        bool(re.search(r'[\u0600-\u06FF]', text)),     # Arabic
        bool(re.search(r'[\u4E00-\u9FFF]', text)),     # CJK
        bool(re.search(r'[\u0900-\u097F]', text)),     # Devanagari
    ]
    return sum(scripts)

feat["alias_script_diversity"] = company["properties.alias"].apply(count_scripts)
feat["has_weak_alias"]         = company["properties.weakAlias"].notna().astype(int)
feat["has_previous_name"]      = company["properties.previousName"].notna().astype(int)


# ═════════════════════════════════════════════════════════════════════════════
# E. TEMPORAL
# ═════════════════════════════════════════════════════════════════════════════
print("\n── E. Temporal ─────────────────────────────────────────────")

incorp_raw = first_value(company["properties.incorporationDate"])
incorp_year = pd.to_datetime(incorp_raw, errors="coerce").dt.year

feat["incorporation_year"]    = incorp_year
feat["company_age_years"]     = CURRENT_YEAR - incorp_year
feat["incorporation_missing"] = incorp_year.isna().astype(int)
feat["has_dissolution_date"]  = company["properties.dissolutionDate"].notna().astype(int)


# ═════════════════════════════════════════════════════════════════════════════
# F. DIRECTORSHIP (aggregated)
# ═════════════════════════════════════════════════════════════════════════════
print("\n── F. Directorship ─────────────────────────────────────────")

directorship["org_id"]    = clean_id(directorship["properties.organization"])
directorship["dir_id"]    = clean_id(directorship["properties.director"])
directorship["role_raw"]  = directorship["properties.role"].fillna("[]")
directorship["status_raw"] = directorship["properties.status"].fillna("").str.lower()

directorship["is_active"]   = directorship["status_raw"].str.contains("appointed|active", na=False).astype(int)
directorship["is_resigned"] = directorship["status_raw"].str.contains("resigned", na=False).astype(int)

def count_unique_roles(role_val):
    try:
        roles = ast.literal_eval(str(role_val))
        return len(set(roles)) if isinstance(roles, list) else 1
    except Exception:
        return 1

directorship["role_count"] = directorship["role_raw"].apply(count_unique_roles)

dir_agg = (
    directorship
    .groupby("org_id")
    .agg(
        director_count          =("dir_id",     "nunique"),
        active_director_count   =("is_active",  "sum"),
        resigned_director_count =("is_resigned", "sum"),
        director_role_diversity =("role_count", "sum"),
    )
    .reset_index()
    .rename(columns={"org_id": "id"})
)

feat = feat.merge(dir_agg, on="id", how="left")
for col in ["director_count", "active_director_count",
            "resigned_director_count", "director_role_diversity"]:
    feat[col] = feat[col].fillna(0).astype(int)

feat["no_directors_found"] = (feat["director_count"] == 0).astype(int)


# ═════════════════════════════════════════════════════════════════════════════
# G. OWNERSHIP (aggregated)
# ═════════════════════════════════════════════════════════════════════════════
print("\n── G. Ownership ────────────────────────────────────────────")

ownership["asset_id"] = clean_id(ownership["properties.asset"])
ownership["owner_id"] = clean_id(ownership["properties.owner"])

# Parse percentage — strip list formatting then coerce to float
def parse_pct(val):
    if pd.isna(val):
        return np.nan
    try:
        parsed = ast.literal_eval(str(val))
        raw = parsed[0] if isinstance(parsed, list) else parsed
        return float(str(raw).replace("%", "").strip())
    except Exception:
        return np.nan

ownership["pct"] = ownership["properties.percentage"].apply(parse_pct)

own_agg = (
    ownership
    .groupby("asset_id")
    .agg(
        owner_count         =("owner_id", "nunique"),
        avg_ownership_pct   =("pct",      "mean"),
        max_ownership_pct   =("pct",      "max"),
    )
    .reset_index()
    .rename(columns={"asset_id": "id"})
)

feat = feat.merge(own_agg, on="id", how="left")
feat["owner_count"]       = feat["owner_count"].fillna(0).astype(int)
feat["has_majority_owner"]    = (feat["max_ownership_pct"] > 50).astype(int)
feat["ownership_pct_missing"] = feat["avg_ownership_pct"].isna().astype(int)
feat["no_owners_found"]       = (feat["owner_count"] == 0).astype(int)

# Round percentage columns to 2dp
feat["avg_ownership_pct"] = feat["avg_ownership_pct"].round(2)
feat["max_ownership_pct"] = feat["max_ownership_pct"].round(2)


# ═════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═════════════════════════════════════════════════════════════════════════════
print("\n── Summary ─────────────────────────────────────────────────")
print(f"  Shape: {feat.shape[0]:,} rows × {feat.shape[1]} columns")
print(f"\n  Columns ({feat.shape[1]}):")
for col in feat.columns:
    null_pct = feat[col].isna().mean() * 100
    print(f"    {col:<35} nulls: {null_pct:4.1f}%")

print(f"\n  Label distribution:")
for tier, label in [(0, "no flag"), (1, "medium risk"), (2, "high risk")]:
    n = (feat["risk_tier"] == tier).sum()
    print(f"    Tier {tier} ({label}): {n:,}  ({n/len(feat):.1%})")


# ═════════════════════════════════════════════════════════════════════════════
# SAVE
# ═════════════════════════════════════════════════════════════════════════════
feat.to_csv(OUTPUT_PATH, index=False)
print(f"\n  Saved → {OUTPUT_PATH}")
print("\nDone.")


── Loading files ───────────────────────────────────────────
  Loaded Company.csv: 234,456 rows
  Loaded Directorship.csv: 130,838 rows
  Loaded Ownership.csv: 227,029 rows

── Building label ──────────────────────────────────────────
  Tier 0 — no flag:     126,612
  Tier 1 — medium risk: 93,875
  Tier 2 — high risk:   13,969

── A. Geography ────────────────────────────────────────────

── B. Company type ─────────────────────────────────────────

── C. Identifier richness ──────────────────────────────────

── D. Alias complexity ─────────────────────────────────────

── E. Temporal ─────────────────────────────────────────────

── F. Directorship ─────────────────────────────────────────

── G. Ownership ────────────────────────────────────────────

── Summary ─────────────────────────────────────────────────
  Shape: 234,456 rows × 39 columns

  Columns (39):
    id                                  nulls:  0.0%
    target                              nulls:  0.0%
    risk_tier   

In [ ]:
"""
Company Risk Dataset — Exploration
====================================
Explores the final feature matrix we built.
Focuses on:
  1. Basic shape and null rates
  2. Label distribution
  3. Feature distributions per risk tier
  4. Correlations with risk tier
  5. Key red flags — how often do high-risk companies have them
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Update this to wherever you saved company_risk_dataset.csv
DATA_PATH = Path("../../data/processed/company_risk_dataset.csv")
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH, low_memory=False)

TIER_LABELS = {0: "no flag", 1: "medium risk", 2: "high risk"}

# ═════════════════════════════════════════════════════════════════════════════
# 1. BASIC SHAPE
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  1. BASIC SHAPE")
print("═"*65)
print(f"\n  Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"\n  {'COLUMN':<35} {'DTYPE':<12} {'NULLS':>8} {'NULL %':>8}")
print(f"  {'-'*35} {'-'*12} {'-'*8} {'-'*8}")
for col in df.columns:
    nulls = df[col].isna().sum()
    pct   = nulls / len(df) * 100
    print(f"  {col:<35} {str(df[col].dtype):<12} {nulls:>8,} {pct:>7.1f}%")


# ═════════════════════════════════════════════════════════════════════════════
# 2. LABEL DISTRIBUTION
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  2. LABEL DISTRIBUTION")
print("═"*65)
for tier, label in TIER_LABELS.items():
    n = (df["risk_tier"] == tier).sum()
    bar = "█" * int(n / len(df) * 40)
    print(f"\n  Tier {tier} — {label:<15} {n:>8,}  ({n/len(df):.1%})  {bar}")


# ═════════════════════════════════════════════════════════════════════════════
# 3. BINARY FEATURES — rate per tier
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  3. BINARY FEATURES — % True per risk tier")
print("═"*65)

binary_cols = [
    "is_high_risk_country", "country_missing", "jurisdiction_country_mismatch",
    "has_sector", "is_holding", "is_trading",
    "has_lei", "has_duns", "has_bic", "has_vat", "has_tax",
    "has_reg", "has_inn", "has_ogrn",
    "has_weak_alias", "has_previous_name",
    "incorporation_missing", "has_dissolution_date",
    "no_directors_found", "no_owners_found",
    "has_majority_owner", "ownership_pct_missing",
]

print(f"\n  {'FEATURE':<35} {'no flag':>10} {'medium':>10} {'high':>10} {'DIFF(hi-no)':>12}")
print(f"  {'-'*35} {'-'*10} {'-'*10} {'-'*10} {'-'*12}")

rows = []
for col in binary_cols:
    if col not in df.columns:
        continue
    rates = {}
    for tier in [0, 1, 2]:
        sub = df[df["risk_tier"] == tier][col]
        rates[tier] = sub.mean() * 100
    diff = rates[2] - rates[0]
    rows.append((col, rates[0], rates[1], rates[2], diff))

# Sort by absolute difference so biggest signals appear first
rows.sort(key=lambda x: abs(x[4]), reverse=True)
for col, r0, r1, r2, diff in rows:
    sign = "+" if diff > 0 else ""
    print(f"  {col:<35} {r0:>9.1f}% {r1:>9.1f}% {r2:>9.1f}% {sign}{diff:>10.1f}%")


# ═════════════════════════════════════════════════════════════════════════════
# 4. NUMERIC FEATURES — median per tier
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  4. NUMERIC FEATURES — median per risk tier")
print("═"*65)

numeric_cols = [
    "identifier_count", "alias_count", "alias_script_diversity",
    "company_age_years", "director_count", "active_director_count",
    "resigned_director_count", "director_role_diversity",
    "owner_count", "avg_ownership_pct", "max_ownership_pct",
]

print(f"\n  {'FEATURE':<35} {'no flag':>10} {'medium':>10} {'high':>10}")
print(f"  {'-'*35} {'-'*10} {'-'*10} {'-'*10}")
for col in numeric_cols:
    if col not in df.columns:
        continue
    vals = []
    for tier in [0, 1, 2]:
        med = df[df["risk_tier"] == tier][col].median()
        vals.append(f"{med:>10.1f}")
    print(f"  {col:<35} {''.join(vals)}")


# ═════════════════════════════════════════════════════════════════════════════
# 5. LEGAL FORM BUCKET — distribution per tier
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  5. LEGAL FORM BUCKET — % of each tier")
print("═"*65)

if "legal_form_bucket" in df.columns:
    buckets = df["legal_form_bucket"].unique()
    print(f"\n  {'BUCKET':<25} {'no flag':>10} {'medium':>10} {'high':>10}")
    print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*10}")
    bucket_rows = []
    for bucket in buckets:
        rates = []
        for tier in [0, 1, 2]:
            sub = df[df["risk_tier"] == tier]
            rate = (sub["legal_form_bucket"] == bucket).mean() * 100
            rates.append(rate)
        bucket_rows.append((bucket, rates))
    bucket_rows.sort(key=lambda x: x[1][2], reverse=True)
    for bucket, rates in bucket_rows:
        print(f"  {bucket:<25} {rates[0]:>9.1f}% {rates[1]:>9.1f}% {rates[2]:>9.1f}%")


# ═════════════════════════════════════════════════════════════════════════════
# 6. COUNTRY — top 15 countries by high-risk count
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  6. COUNTRY — top 15 by high-risk company count")
print("═"*65)

if "country" in df.columns:
    high_risk = df[df["risk_tier"] == 2]
    top = high_risk["country"].value_counts().head(15)
    print(f"\n  {'COUNTRY':<12} {'HIGH RISK COUNT':>18} {'% OF HIGH RISK':>16}")
    print(f"  {'-'*12} {'-'*18} {'-'*16}")
    for country, count in top.items():
        pct = count / len(high_risk) * 100
        print(f"  {str(country):<12} {count:>18,} {pct:>15.1f}%")


# ═════════════════════════════════════════════════════════════════════════════
# 7. CORRELATION WITH RISK TIER
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  7. CORRELATION WITH RISK TIER (numeric cols only)")
print("═"*65)

num_df = df.select_dtypes(include=[np.number]).drop(columns=["risk_tier"], errors="ignore")
corrs  = num_df.corrwith(df["risk_tier"]).sort_values(key=abs, ascending=False)

print(f"\n  {'FEATURE':<35} {'CORRELATION':>12}")
print(f"  {'-'*35} {'-'*12}")
for col, corr in corrs.items():
    if pd.notna(corr):
        print(f"  {col:<35} {corr:>+12.4f}")

print("\n" + "═"*65)
print("  Done. Paste output back to review findings.")
print("═"*65)